# 06. 멀티모달 검색 실험 — Basic vs. Verbalized 비교

이 노트북에서는 `05-multimodal-indexing.ipynb`에서 생성된 두 인덱스를 대상으로  
검색 품질을 비교하고 RAG 실험을 수행합니다.

## 비교 대상 인덱스

| 인덱스 | 파이프라인 | 특징 |
|--------|-----------|------|
| `st-multimodal-basic-index` | [B] DI Layout → SplitSkill → Embedding | 텍스트 위주, 빠른 인덱싱 |
| `st-multimodal-verbalized-index` | [C] DI Layout → GPT-5.4 Verbalize → Split → Embedding | 이미지/도표 의미 설명 포함, 풍부한 컨텍스트 |

## 실험 항목
1. **키워드 검색** — 양쪽 인덱스에서 동일 쿼리로 검색
2. **벡터 검색** — 시맨틱 유사도 기반 비교
3. **하이브리드 검색** — 키워드 + 벡터 + 시맨틱 랭커
4. **RAG (Retrieval-Augmented Generation)** — GPT-5.4로 응답 생성 품질 비교
5. **이미지/도표 질문** — Verbalized 파이프라인의 강점 검증

## 1. 환경 설정

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

import requests
from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI

credential = DefaultAzureCredential()

SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
GPT_DEPLOYMENT = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')
EMBEDDING_DEPLOYMENT = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')

# 인덱스 이름
SOURCE = 'st'
BASIC_INDEX = f'{SOURCE}-multimodal-basic-index'
VERBALIZED_INDEX = f'{SOURCE}-multimodal-verbalized-index'

API_VERSION = '2024-11-01-preview'

print('환경 설정 완료')
print(f'  AI Search : {SEARCH_ENDPOINT}')
print(f'  OpenAI    : {OPENAI_ENDPOINT}')
print(f'  GPT       : {GPT_DEPLOYMENT}')
print(f'  Embedding : {EMBEDDING_DEPLOYMENT}')
print(f'  인덱스 B  : {BASIC_INDEX}')
print(f'  인덱스 C  : {VERBALIZED_INDEX}')

In [ ]:
# ── 클라이언트 초기화 ─────────────────────────────────────────

def get_search_headers():
    token = credential.get_token('https://search.azure.com/.default').token
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}


def get_openai_client() -> AzureOpenAI:
    token = credential.get_token('https://cognitiveservices.azure.com/.default').token
    return AzureOpenAI(
        azure_endpoint=OPENAI_ENDPOINT,
        api_version='2024-12-01-preview',
        azure_ad_token=token,
    )


def get_embedding(text: str) -> list[float]:
    """텍스트 임베딩 생성"""
    client = get_openai_client()
    response = client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=text,
    )
    return response.data[0].embedding


print('클라이언트 초기화 완료')

## 2. 인덱스 통계 확인

In [ ]:
def get_index_stats(index_name: str) -> dict:
    url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/stats?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    return resp.json() if resp.status_code == 200 else {}

print(f'{"인덱스":<40} {"문서 수":>10} {"저장 크기 (KB)":>15}')
print('─'*70)

for idx_name in [BASIC_INDEX, VERBALIZED_INDEX]:
    stats = get_index_stats(idx_name)
    doc_count = stats.get('documentCount', 'N/A')
    storage_kb = stats.get('storageSize', 0) // 1024
    print(f'{idx_name:<40} {doc_count:>10} {storage_kb:>12,} KB')

## 3. 검색 유틸리티 함수

두 인덱스에 대해 동일한 검색을 수행하고 결과를 비교하는 함수들을 정의합니다.

In [ ]:
def search_index(
    index_name: str,
    query: str = '',
    vector: list[float] | None = None,
    top: int = 5,
    semantic_config: str = 'mm-semantic-config',
    query_type: str = 'simple',
    select: str = 'id,source_file_name,content_type,content',
) -> list[dict]:
    """AI Search 쿼리 실행"""
    url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/docs/search?api-version={API_VERSION}'
    
    payload = {
        'top': top,
        'select': select,
        'count': True,
    }
    
    if query:
        payload['search'] = query
        payload['queryType'] = query_type
    
    if query_type == 'semantic':
        payload['semanticConfiguration'] = semantic_config
    
    if vector:
        payload['vectorQueries'] = [{
            'kind': 'vector',
            'vector': vector,
            'fields': 'content_vector',
            'k': top,
        }]
    
    resp = requests.post(url, headers=get_search_headers(), json=payload)
    if resp.status_code != 200:
        print(f'  ERROR: {resp.status_code} {resp.text[:200]}')
        return []
    
    data = resp.json()
    results = data.get('value', [])
    return results


def compare_search(
    query: str,
    vector: list[float] | None = None,
    top: int = 5,
    query_type: str = 'simple',
    show_content_len: int = 150,
):
    """두 인덱스에 동일 쿼리 실행 후 결과 비교 출력"""
    print(f'\n{"═"*70}')
    print(f'  Query: "{query}"  (type={query_type}, vector={vector is not None})')
    print(f'{"═"*70}')
    
    for label, idx_name in [('[B] Basic', BASIC_INDEX), ('[C] Verbalized', VERBALIZED_INDEX)]:
        results = search_index(idx_name, query=query, vector=vector, top=top, query_type=query_type)
        print(f'\n  ── {label} ({idx_name}) ── {len(results)}건')
        for i, doc in enumerate(results):
            score = doc.get('@search.score', 0)
            content = doc.get('content', '')
            fname = doc.get('source_file_name', '?')
            ctype = doc.get('content_type', '?')
            snippet = content[:show_content_len].replace('\n', ' ')
            print(f'    {i+1}. [{ctype}] {fname} (score={score:.4f})')
            print(f'       {snippet}...' if len(content) > show_content_len else f'       {snippet}')


print('검색 유틸리티 함수 정의 완료')

## 4. 실험 1 — 키워드 검색 비교

In [ ]:
# 키워드 검색 테스트
test_queries_keyword = [
    'single line diagram',
    '전기 설비 구성',
    '변압기 사양',
]

for q in test_queries_keyword:
    compare_search(q, query_type='simple')

## 5. 실험 2 — 벡터 검색 비교

In [ ]:
# 벡터 검색 테스트
test_queries_vector = [
    '전력 계통 단선도의 주요 구성 요소와 연결 관계',
    '변전소 설비 용량 및 사양 정보',
    '도면에 표시된 보호 계전기 동작 특성',
]

for q in test_queries_vector:
    embedding = get_embedding(q)
    compare_search(q, vector=embedding, query_type='simple')

## 6. 실험 3 — 하이브리드 + 시맨틱 검색 비교

키워드 + 벡터 + Semantic Ranker를 결합한 하이브리드 검색으로 비교합니다.

In [ ]:
test_queries_hybrid = [
    '이 문서에서 설명하는 전기 시스템의 전체 구조는?',
    '도면/다이어그램에 나타난 핵심 정보를 요약해주세요',
    '설비 보호 방식과 차단기 동작 조건',
]

for q in test_queries_hybrid:
    embedding = get_embedding(q)
    compare_search(q, vector=embedding, query_type='semantic')

## 7. 실험 4 — RAG 비교 (GPT-5.4 응답 생성)

두 인덱스에서 각각 Top-K 결과를 가져온 후, GPT-5.4로 응답을 생성하여 품질을 비교합니다.

In [ ]:
def rag_answer(
    question: str,
    index_name: str,
    top_k: int = 5,
) -> str:
    """RAG: 검색 → GPT-5.4 응답 생성"""
    # 1. 임베딩 + 하이브리드 검색
    embedding = get_embedding(question)
    results = search_index(
        index_name, query=question, vector=embedding,
        top=top_k, query_type='semantic',
    )
    
    if not results:
        return '검색 결과 없음'
    
    # 2. 컨텍스트 구성
    context_parts = []
    for i, doc in enumerate(results):
        content = doc.get('content', '')
        fname = doc.get('source_file_name', '?')
        ctype = doc.get('content_type', 'text')
        context_parts.append(f'[{i+1}] ({fname}, type={ctype})\n{content}')
    
    context = '\n\n'.join(context_parts)
    
    # 3. GPT-5.4 응답 생성
    client = get_openai_client()
    response = client.chat.completions.create(
        model=GPT_DEPLOYMENT,
        messages=[
            {
                'role': 'system',
                'content': (
                    '주어진 컨텍스트를 기반으로 질문에 답변하세요. '
                    '답변에는 출처 번호([1], [2] 등)를 포함하여 근거를 명시하세요. '
                    '컨텍스트에 없는 내용은 추측하지 마세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'컨텍스트:\n{context}\n\n질문: {question}',
            },
        ],
        max_tokens=1500,
        temperature=0.3,
    )
    
    return response.choices[0].message.content


print('RAG 함수 정의 완료')

In [ ]:
# ── RAG 비교 실험 ─────────────────────────────────────────────
rag_questions = [
    '이 문서에 포함된 단선도(Single Line Diagram)의 주요 구성 요소와 그 역할을 설명해주세요.',
    '문서에 나타난 다이어그램이나 도면에서 어떤 기술적 정보를 확인할 수 있나요?',
    '전기 설비의 보호 방식과 관련된 내용을 정리해주세요.',
]

for question in rag_questions:
    print(f'\n{"═"*70}')
    print(f'  질문: {question}')
    print(f'{"═"*70}')
    
    for label, idx_name in [('[B] Basic', BASIC_INDEX), ('[C] Verbalized', VERBALIZED_INDEX)]:
        print(f'\n  ── {label} 응답 ──')
        answer = rag_answer(question, idx_name)
        print(f'  {answer}')
    
    print()

## 8. 실험 5 — 이미지/도표 전용 질문 (Verbalized 강점)

시각적 정보(다이어그램, 도면, 차트)에 대한 질문으로  
Verbalized 파이프라인의 이점을 검증합니다.

In [ ]:
visual_questions = [
    '단선도에 표시된 차단기(CB)와 변압기(TR)의 연결 관계를 설명해주세요.',
    '도면에서 확인할 수 있는 전압 등급별 모선(Bus) 구성은 어떻게 되나요?',
    '다이어그램에 나타난 보호 계전기의 종류와 설치 위치는?',
]

for question in visual_questions:
    print(f'\n{"═"*70}')
    print(f'  [이미지/도표 질문] {question}')
    print(f'{"═"*70}')
    
    for label, idx_name in [('[B] Basic', BASIC_INDEX), ('[C] Verbalized', VERBALIZED_INDEX)]:
        print(f'\n  ── {label} 응답 ──')
        answer = rag_answer(question, idx_name)
        print(f'  {answer}')
    
    print()

## 9. 정량 비교 요약

청크 수, 검색 적합성, 응답 품질을 정량적으로 비교합니다.

In [ ]:
# ── 인덱스 비교 요약 테이블 ───────────────────────────────────
print(f'\n{"═"*70}')
print('멀티모달 파이프라인 비교 요약')
print(f'{"═"*70}')
print(f'\n{"항목":<25} {"[B] Basic":<25} {"[C] Verbalized":<25}')
print('─'*75)

# 문서 수 / 크기
for idx_name, label in [(BASIC_INDEX, '[B] Basic'), (VERBALIZED_INDEX, '[C] Verbalized')]:
    stats = get_index_stats(idx_name)
    doc_count = stats.get('documentCount', '?')
    storage_kb = stats.get('storageSize', 0) // 1024
    print(f'  {label}: 문서 {doc_count}개, 크기 {storage_kb:,} KB')

print()
print('비교 포인트:')
print('  1. Basic은 SplitSkill 네이티브 markdown 모드 → 빠르고 비용 없음')
print('  2. Verbalized는 GPT-5.4가 이미지/도표를 자연어로 설명 → 풍부한 컨텍스트')
print('  3. 텍스트 위주 문서: 두 파이프라인 성능 유사')
print('  4. 이미지/도면 많은 문서: Verbalized 우위 (도표 의미 설명 포함)')
print('  5. 비용: Basic 무료 vs Verbalized GPT 호출 비용')
print()
print('권장 사용처:')
print('  [B] Basic     → 텍스트 위주 문서 (보고서, 논문, 법률문서 등)')
print('  [C] Verbalized → 이미지/도면 포함 문서 (기술도면, 설계서, 프레젠테이션 등)')

---

## 정리

| 실험 | 검색 방식 | 핵심 비교 포인트 |
|------|----------|------------------|
| 실험 1 | 키워드 검색 | 텍스트 매칭 기반 — 두 인덱스 차이 적음 |
| 실험 2 | 벡터 검색 | 시맨틱 유사도 — Verbalized가 도표 설명 텍스트로 유리 |
| 실험 3 | 하이브리드 + 시맨틱 | 최적 검색 방식 — Verbalized 시맨틱 랭킹 유리 |
| 실험 4 | RAG 응답 생성 | 답변 품질 — 컨텍스트 풍부한 Verbalized 우위 |
| 실험 5 | 이미지/도표 질문 | Verbalized 핵심 가치 검증 |

### 결론
- **텍스트 위주 문서**: Basic으로 충분 (비용 효율적)
- **이미지/도면 포함 문서**: Verbalized 필수 (GPT-5.4 이미지 설명이 RAG 품질 향상)
- **실무**: 문서 유형에 따라 두 파이프라인을 선택적으로 적용하는 것을 권장